In [24]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [25]:
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k = None, device = "mps")
pipe("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528688922524452},
  {'label': 'neutral', 'score': 0.005764603149145842},
  {'label': 'anger', 'score': 0.004419787786900997},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [26]:
pipe(books["description"][0])

[[{'label': 'fear', 'score': 0.6548410058021545},
  {'label': 'neutral', 'score': 0.16985207796096802},
  {'label': 'sadness', 'score': 0.1164090484380722},
  {'label': 'surprise', 'score': 0.02070067822933197},
  {'label': 'disgust', 'score': 0.019100744277238846},
  {'label': 'joy', 'score': 0.015161268413066864},
  {'label': 'anger', 'score': 0.0039351521991193295}]]

In [27]:
sentences = books["description"][0].split(".")
predictions = pipe(sentences)

In [28]:
predictions

[[{'label': 'surprise', 'score': 0.7296029329299927},
  {'label': 'neutral', 'score': 0.1403854936361313},
  {'label': 'fear', 'score': 0.06816215813159943},
  {'label': 'joy', 'score': 0.047942403703927994},
  {'label': 'anger', 'score': 0.009156346321105957},
  {'label': 'disgust', 'score': 0.0026284733321517706},
  {'label': 'sadness', 'score': 0.0021221591159701347}],
 [{'label': 'neutral', 'score': 0.44937166571617126},
  {'label': 'disgust', 'score': 0.27359047532081604},
  {'label': 'joy', 'score': 0.10908303409814835},
  {'label': 'sadness', 'score': 0.09362732619047165},
  {'label': 'anger', 'score': 0.04047819599509239},
  {'label': 'surprise', 'score': 0.026970207691192627},
  {'label': 'fear', 'score': 0.00687905540689826}],
 [{'label': 'neutral', 'score': 0.6462162137031555},
  {'label': 'sadness', 'score': 0.24273322522640228},
  {'label': 'disgust', 'score': 0.04342261329293251},
  {'label': 'surprise', 'score': 0.02830052375793457},
  {'label': 'joy', 'score': 0.0142114

In [29]:
predictions[3]

[{'label': 'fear', 'score': 0.9281686544418335},
 {'label': 'anger', 'score': 0.03219055011868477},
 {'label': 'neutral', 'score': 0.012808658182621002},
 {'label': 'sadness', 'score': 0.008756828494369984},
 {'label': 'surprise', 'score': 0.00859787967056036},
 {'label': 'disgust', 'score': 0.008431786671280861},
 {'label': 'joy', 'score': 0.0010455787414684892}]

In [30]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009156346321105957},
 {'label': 'disgust', 'score': 0.0026284733321517706},
 {'label': 'fear', 'score': 0.06816215813159943},
 {'label': 'joy', 'score': 0.047942403703927994},
 {'label': 'neutral', 'score': 0.1403854936361313},
 {'label': 'sadness', 'score': 0.0021221591159701347},
 {'label': 'surprise', 'score': 0.7296029329299927}]

In [31]:
import numpy as np

emotional_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
isbn           = []
emotion_scores = {label: [] for label in emotional_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotional_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x : x["label"])
        for index, label in enumerate(emotional_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])

    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [32]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = pipe(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotional_labels:
        emotion_scores[label].append(max_scores[label])


In [33]:
emotion_scores

{'anger': [np.float64(0.06413350254297256),
  np.float64(0.6126184463500977),
  np.float64(0.06413350254297256),
  np.float64(0.35148346424102783),
  np.float64(0.08141236007213593),
  np.float64(0.23222514986991882),
  np.float64(0.5381848812103271),
  np.float64(0.06413350254297256),
  np.float64(0.30067020654678345),
  np.float64(0.06413350254297256)],
 'disgust': [np.float64(0.27359047532081604),
  np.float64(0.3482857048511505),
  np.float64(0.10400652885437012),
  np.float64(0.15072233974933624),
  np.float64(0.18449483811855316),
  np.float64(0.7271745800971985),
  np.float64(0.15585485100746155),
  np.float64(0.10400652885437012),
  np.float64(0.27948108315467834),
  np.float64(0.1779266744852066)],
 'fear': [np.float64(0.9281686544418335),
  np.float64(0.942527711391449),
  np.float64(0.9723208546638489),
  np.float64(0.36070725321769714),
  np.float64(0.09504333883523941),
  np.float64(0.05136270821094513),
  np.float64(0.7474287152290344),
  np.float64(0.40449631214141846),


In [34]:
from tqdm import tqdm

emotional_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
isbn           = []
emotion_scores = {label: [] for label in emotional_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = pipe(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotional_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [4:03:12<00:00,  2.81s/it]    


In [35]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [36]:
emotions_df.head()

,anger,disgust,fear,joy,sadness,surprise,isbn13
0,0.064134,0.273590,0.928169,0.932798,0.646216,0.967158,9780002005883
1,0.612618,0.348286,0.942528,0.704422,0.887940,0.111690,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,9780006178736
3,0.351483,0.150722,0.360707,0.251881,0.732686,0.111690,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.884389,0.475881,9780006280934


In [37]:
books = pd.merge(books,emotions_df, on="isbn13")

In [38]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273590,0.928169,0.932798,0.646216,0.967158
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612618,0.348286,0.942528,0.704422,0.887940,0.111690
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.150722,0.360707,0.251881,0.732686,0.111690
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.884389,0.475881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.148209,0.030643,0.919165,0.255170,0.853722,0.980877
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.064134,0.114383,0.051363,0.400263,0.883198,0.111690
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339216,0.947779,0.375757,0.066685
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.064134,0.104007,0.459270,0.759455,0.951104,0.368111


In [39]:
books.to_csv(
    "books_with_emotions.csv",
    index=False,
)